# 📚 처음 시작하는 사람을 위한 판단 가이드

이 파일 하나면 아래 세 가지를 모두 결정할 수 있어요!

| 번호 | 결정해야 할 것 | 설명 |
|------|--------------|------|
| 1 | **분류 vs 회귀** | 어떤 템플릿 파일을 열면 되나? |
| 2 | **컬럼 처리** | 각 컬럼을 어떻게 다뤄야 하나? |
| 3 | **하이퍼파라미터** | 튜닝할 때 어떤 숫자를 넣으면 되나? |

---

# PART 1. 분류 vs 회귀 — 어떤 템플릿을 쓸까?

## 🍎 핵심은 딱 하나! "정답(Target)이 무엇의 형태인가?"

```
친구에게 물어본다고 생각해요.

  "이 사람 살았어, 죽었어?"  → 답이 '살았어' 또는 '죽었어'  → 분류
  "이 택시 몇 초 걸렸어?"    → 답이 '423초'처럼 숫자          → 회귀
```

## 한눈에 보는 차이

| | 분류 (Classification) | 회귀 (Regression) |
|---|---|---|
| **정답 예시** | 0 또는 1, A/B/C, 살았다/죽었다 | 423초, 15만원, 37.5도 |
| **"이게 뭐야?" 질문** | "몇 종류 중 하나야?" | "얼마야? 몇이야?" |
| **평가 기준** | accuracy (몇 % 맞혔나) | RMSE, MAE (얼마나 차이나나) |
| **사용 파일** | `분류_Classification_템플릿.ipynb` | `회귀_Regression_템플릿.ipynb` |

## 📋 판단하는 3단계

### 1단계: 문제 설명에서 Target 찾기

```
"생존 여부를 예측"      → 생존/사망 → 분류
"등급을 예측"           → A/B/C    → 분류  
"이동 시간을 예측"      → 몇 초    → 회귀
"대여 건수를 예측"      → 몇 개    → 회귀
"집값을 예측"           → 얼마     → 회귀
```

### 2단계: 평가지표 확인

```
accuracy, f1_score, roc_auc  → 분류
RMSE, MAE, R², RMSLE         → 회귀
```

### 3단계: 데이터로 직접 확인 (아래 코드 실행)

In [ ]:
import pandas as pd
import numpy as np

# train = pd.read_csv('./data/train.csv')

def check_problem_type(df, target_col):
    """
    Target 컬럼을 보고 분류인지 회귀인지 알려줍니다.
    """
    s = df[target_col]
    print(f'[ Target: {target_col} ]')
    print(f'  데이터 타입  : {s.dtype}')
    print(f'  서로 다른 값 수 : {s.nunique()}개')
    print(f'  값 예시      : {sorted(s.dropna().unique()[:8])}')
    print()

    if s.dtype == 'object':
        print('→ ✅ 분류 문제  (글자 타입 → 몇 가지 종류 중 하나)')
        print('   파일: 분류_Classification_템플릿.ipynb')
    elif s.nunique() <= 20:
        print(f'→ ✅ 분류 문제 가능성 높음  (값 종류가 {s.nunique()}개로 적음)')
        print('   파일: 분류_Classification_템플릿.ipynb')
    else:
        print(f'→ ✅ 회귀 문제  (값 종류가 {s.nunique()}개로 많음 → 연속된 숫자)')
        print('   파일: 회귀_Regression_템플릿.ipynb')

# ★ TARGET을 실제 target 컬럼명으로 바꾸고 실행!
# check_problem_type(train, 'Survived')

## 수업 예제 정리

| 과제 | Target 컬럼 | 값 예시 | 판정 |
|------|------------|--------|------|
| Titanic 생존 예측 | Survived | 0 (사망), 1 (생존) | ✅ 분류 |
| SOH 배터리 상태 | SOH등급 | A, B, C, D | ✅ 분류 |
| 자전거 대여량 | count | 0, 13, 47, 231... | ✅ 회귀 |
| 택시 이동시간 | trip_duration | 32, 423, 1205... | ✅ 회귀 |

---

# PART 2. 컬럼 처리 — 어떤 컬럼을 어떻게 다루나?

## 🧹 전처리가 필요한 이유

```
컴퓨터(AI 모델)는 오직 '숫자'만 이해합니다.

그래서:
  - '남성', '여성' 같은 글자  → 숫자로 바꿔줘야 함  (OHE)
  - 175cm, 75kg처럼 단위가 다른 숫자 → 같은 단위로 맞춰줘야 함  (스케일링)
  - 이름, ID처럼 쓸모없는 것  → 지워야 함  (삭제)
```

## 컬럼 4종류와 처리 방법

| 기호 | 종류 | 예시 | 처리 방법 |
|------|------|------|----------|
| ❌ | **삭제** | ID, 이름, 주소 | `drop()` — 그냥 버림 |
| 🔤 | **질적변수** (범주형) | 성별, 등급, 계절 | `get_dummies()` — 숫자로 번역 |
| 📊 | **양적변수** (연속형) | 나이, 요금, 거리 | `MinMaxScaler()` — 단위 통일 |
| 📅 | **날짜형** | 2016-01-18 11:10 | 월/일/시간/요일로 분리 후 🔤 처리 |

## ❌ 삭제 대상 — 이런 컬럼은 지우세요

```
✅ 삭제해야 하는 것들:

  PassengerId, id → 그냥 순서 번호. 예측에 도움 안 됨
  Name (이름)     → 사람마다 다 달라서 패턴 없음
  Ticket (티켓번호) → 의미 없는 코드
  Cabin (객실번호) → 결측치가 너무 많음
  dropoff_datetime → test 데이터에 없는 컬럼은 쓸 수 없음!

💡 기준: test 데이터에 없는 컬럼 = 무조건 삭제
         고유값 수 = 전체 행 수인 컬럼 = 거의 삭제 (이름, ID류)
```

## 🔤 질적변수 OHE — 왜 하나요?

```
컴퓨터는 '남성' '여성'이라는 글자를 모릅니다.
그래서 이렇게 바꿔줍니다:

  원래:  Sex = 'male'    → Sex_male=1, Sex_female=0
  원래:  Sex = 'female'  → Sex_male=0, Sex_female=1

pd.get_dummies()가 이걸 자동으로 해줍니다!

어떤 컬럼이 질적변수?
  - dtype이 object (글자형) → 무조건 질적변수
  - dtype이 숫자인데 고유값이 10개 이하 → 질적변수일 가능성 높음
    예) Pclass: 1, 2, 3 (딱 3종류)  → 질적변수
        vendor_id: 1, 2 (딱 2종류) → 질적변수
```

## 📊 양적변수 스케일링 — 왜 하나요?

```
나이(Age)는 1~80,  요금(Fare)은 0~512  → 범위가 너무 다릅니다.

컴퓨터는 숫자가 크면 "더 중요하다"고 오해할 수 있어요.
그래서 모든 숫자를 0~1 사이로 맞춰줍니다.

  Age=22  → 0.27  (22를 0~80 기준으로 정규화)
  Fare=7  → 0.01  (7을 0~512 기준으로 정규화)

어떤 컬럼이 양적변수?
  - dtype이 float64 → 거의 항상 양적변수
  - dtype이 int인데 고유값이 많음 (11개 이상) → 양적변수
    예) Age(나이): 0~80의 정수  → 양적변수
```

In [ ]:
# 🔍 이 코드를 실행하면 각 컬럼을 어떻게 처리할지 자동으로 알려줍니다!

def analyze_columns(df, target_col, drop_candidates=None):
    """
    df              : train 데이터프레임
    target_col      : 정답 컬럼 이름 (문자열)
    drop_candidates : 미리 알고 있는 삭제 대상 컬럼 리스트
    """
    if drop_candidates is None:
        drop_candidates = []

    print(f"{'컬럼명':22} {'타입':10} {'고유값수':>8} {'결측수':>6}  처리 방향")
    print('-' * 80)

    for col in df.columns:
        if col == target_col:
            print(f"{col:22} {'':10} {'':>8} {'':>6}  🎯 TARGET (정답 컬럼 — 건드리지 않음)")
            continue
        dtype    = str(df[col].dtype)
        n_unique = df[col].nunique()
        n_null   = df[col].isnull().sum()

        if col in drop_candidates:
            direction = '❌ 삭제  (ID/이름류 또는 test에 없는 컬럼)'
        elif dtype == 'object':
            if n_unique <= 20:
                direction = '🔤 OHE  (글자형 → 숫자로 번역 필요)'
            else:
                direction = '❌ 삭제 검토  (고유값 너무 많음 → 이름/텍스트류)'
        elif 'datetime' in dtype:
            direction = '📅 날짜 분리  (월/일/시/요일 추출 후 OHE)'
        elif n_unique <= 10:
            direction = f'🔤 OHE 검토  (숫자지만 {n_unique}가지 밖에 없음 → 카테고리 가능)'
        else:
            direction = '📊 스케일링  (연속 숫자 → MinMaxScaler 적용)'

        null_mark = f'  ⚠️ 결측치 있음! ({n_null}개 — 채워줘야 함)' if n_null > 0 else ''
        print(f"{col:22} {dtype:10} {n_unique:>8} {n_null:>6}  {direction}{null_mark}")

    print()
    print('📌 결측치 채우는 방법:')
    print('   📊 양적변수(숫자) → 평균값으로 채우기: fillna(train[col].mean())')
    print('   🔤 질적변수(글자) → 가장 많은 값으로: fillna(train[col].mode()[0])')


# ★ 아래 두 줄을 실제 문제에 맞게 수정하고 실행!
# analyze_columns(
#     train,
#     target_col='Survived',
#     drop_candidates=['PassengerId', 'Name', 'Ticket', 'Cabin']
# )

## 스케일러 선택 — 어떤 것을 써야 하나?

```
수업에서는 항상 MinMaxScaler 씁니다. 이것만 기억하면 됩니다!

MinMaxScaler: 모든 숫자를 0과 1 사이로 변환
  예) 최솟값 0, 최댓값 512인 Fare에서
      Fare=7   → (7-0)/(512-0)   = 0.014
      Fare=256 → (256-0)/(512-0) = 0.5
      Fare=512 → (512-0)/(512-0) = 1.0

⚠️ 중요한 규칙:
  fit_transform(train) → train 기준으로 0~1 범위 계산 + 변환
  transform(test)      → train 기준으로 변환만 (계산 다시 안 함!)

  왜? test 데이터를 먼저 보면 "미래를 미리 아는 것"이라 안 됨!
```

---

# PART 3. 하이퍼파라미터 튜닝 — 어떤 숫자를 넣으면 되나?

## 🍳 하이퍼파라미터란?

```
요리 레시피를 생각해보세요.

  "설탕 2스푼, 소금 0.5스푼" → 이게 하이퍼파라미터

  설탕을 1스푼 넣어봤더니 맛이 없고,
  3스푼 넣어봤더니 너무 달고,
  2스푼이 딱 맞더라!

  → AI 모델도 똑같이 여러 값을 넣어보고 가장 좋은 것을 찾습니다.
  → 이것이 GridSearchCV (격자 탐색)입니다!
```

## 모델별 넣는 값 (수업 기준)

### 🥇 LGBM (가장 자주 최선)
```python
parameters = {
    'learning_rate': [0.05, 0.1, 0.14],  # 학습 속도 (작을수록 천천히, 정교하게)
    'n_estimators':  [100, 200, 300]      # 트리 개수 (많을수록 꼼꼼하게)
}
```

### 🥈 XGBoost
```python
parameters = {
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators':  [100, 200, 300],
    'max_depth':     [3, 5, 7]            # 트리 깊이 (깊을수록 복잡한 패턴 학습)
}
```

### 🥉 RandomForest / ExtraTrees
```python
parameters = {
    'n_estimators': [100, 200, 300],
    'max_depth':    [None, 5, 10]         # None = 제한 없음
}
```

## ⏱️ 시간이 얼마나 걸리나?

```
조합 수 = 파라미터1 후보 수 × 파라미터2 후보 수 × cv(교차검증 횟수)

예) learning_rate 3개 × n_estimators 3개 × cv=5 = 45번 학습!
    → 데이터가 크면 오래 걸림. 처음엔 작은 범위로 시작하세요.
```

In [ ]:
# 🔍 이 코드를 실행하면 선택된 모델에 맞는 파라미터를 자동으로 알려줍니다!

PARAM_GUIDE = {
    'LGBMClassifier':        {'learning_rate': [0.05, 0.1, 0.14], 'n_estimators': [100, 200, 300]},
    'LGBMRegressor':         {'learning_rate': [0.05, 0.1, 0.14], 'n_estimators': [100, 200, 300]},
    'XGBClassifier':         {'learning_rate': [0.05, 0.1, 0.2],  'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 7]},
    'XGBRegressor':          {'learning_rate': [0.05, 0.1, 0.2],  'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 7]},
    'RandomForestClassifier':{'n_estimators': [100, 200, 300], 'max_depth': [None, 5, 10]},
    'RandomForestRegressor': {'n_estimators': [100, 200, 300], 'max_depth': [None, 5, 10]},
    'ExtraTreesClassifier':  {'n_estimators': [100, 200, 300], 'max_depth': [None, 5, 10]},
    'ExtraTreesRegressor':   {'n_estimators': [100, 200, 300], 'max_depth': [None, 5, 10]},
    'DecisionTreeClassifier':{'max_depth': [3, 5, 7, 10, None], 'min_samples_split': [2, 5, 10]},
    'DecisionTreeRegressor': {'max_depth': [3, 5, 7, 10, None], 'min_samples_split': [2, 5, 10]},
    'KNeighborsClassifier':  {'n_neighbors': [3, 5, 7, 9]},
}

def get_param_guide(model):
    name = type(model).__name__
    if name in PARAM_GUIDE:
        params = PARAM_GUIDE[name]
        print(f'✅ [{name}] 권장 파라미터 후보:')
        print()
        print('parameters = {')
        for k, v in params.items():
            print(f"    '{k}': {v},")
        print('}')
        n_combo = 1
        for v in params.values():
            n_combo *= len(v)
        print(f'\n→ cv=5 기준 총 {n_combo * 5}번 학습 (조합 {n_combo}개 × 5)')
        print('→ 위 코드를 GridSearchCV의 param_grid에 그대로 복붙하세요!')
    else:
        print(f'[{name}]: 파라미터 가이드 없음. parameters = {{}} 로 기본값 사용')

# ★ 모델 for문이 끝난 뒤 아래를 실행:
# get_param_guide(selected_estimator)

---

# 📌 전체 흐름 요약 — 이 순서대로 따라하면 됩니다!

```
STEP 1: 분류? 회귀?
  → check_problem_type(train, 'Target컬럼명') 실행
  → 분류면 분류_Classification_템플릿.ipynb 열기
  → 회귀면  회귀_Regression_템플릿.ipynb 열기

STEP 2: 어떤 컬럼을 어떻게?
  → analyze_columns(train, 'Target컬럼명', drop_candidates=[...]) 실행
  → 출력 보고 ❌ / 🔤 / 📊 / 📅 로 분류해서 해당 리스트에 넣기

STEP 3: 결측치 채우기
  → ⚠️ 표시된 컬럼 확인
  → 📊 양적변수: fillna(평균)
  → 🔤 질적변수: fillna(최빈값)
  → 항상 train 기준으로 계산, test에는 적용만!

STEP 4: 아웃라이어 제거
  → 회귀면 Target도 포함! IQR 방법 사용
  → 분류는 Target 건드리지 않음 (0/1 값이라 이상치 없음)

STEP 5: 모델 for문 실행 → 최선 모델 확인
  → get_param_guide(selected_estimator) 실행
  → 출력된 파라미터를 GridSearchCV에 복붙

STEP 6: 예측 후 제출 파일 생성
```

---

## 🚨 자주 하는 실수 TOP 5

```
1. test에 fit_transform() 사용 ❌  →  transform() 만 해야 함
2. Target 컬럼을 feature에 포함   ❌  →  drop([TARGET]) 먼저!
3. train/test 둘 다 결측치 처리했는데 train 기준이 아님  ❌
4. 회귀에서 log 변환 후 예측했는데 expm1() 역변환 안 함  ❌
5. 분류인데 accuracy 대신 RMSE 사용  ❌  →  문제 유형 다시 확인!
```